In [1]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


In [2]:
import os

path = r"C:\Users\PRAJYOT\OneDrive\Desktop\uidai projrct"
os.path.exists(path)


True

In [5]:
import os
import pandas as pd

BASE_PATH = r"C:\Users\PRAJYOT\OneDrive\Desktop\uidai projrct"

enrolment_files = [
    os.path.join(BASE_PATH, "api_data_aadhar_enrolment_0_500000.csv"),
    os.path.join(BASE_PATH, "api_data_aadhar_enrolment_500000_1000000.csv")
]

# Test first file
pd.read_csv(enrolment_files[0]).head()

# -------- DEMOGRAPHIC DATA --------
demo_files = [
    os.path.join(BASE_PATH, "api_data_aadhar_demographic_0_500000.csv"),
    os.path.join(BASE_PATH, "api_data_aadhar_demographic_500000_1000000.csv"),
    os.path.join(BASE_PATH, "api_data_aadhar_demographic_1000000_1500000.csv"),
    os.path.join(BASE_PATH, "api_data_aadhar_demographic_2000000_2071700.csv")
]

# Test first demographic file
pd.read_csv(demo_files[0]).head()


# -------- BIOMETRIC DATA --------
bio_files = [
    os.path.join(BASE_PATH, "api_data_aadhar_biometric_0_500000.csv"),
    os.path.join(BASE_PATH, "api_data_aadhar_biometric_500000_1000000.csv"),
    os.path.join(BASE_PATH, "api_data_aadhar_biometric_1000000_1500000.csv")
]

# Test first biometric file
pd.read_csv(bio_files[0]).head()

enrol_df = pd.concat([pd.read_csv(f) for f in enrolment_files], ignore_index=True)
demo_df  = pd.concat([pd.read_csv(f) for f in demo_files], ignore_index=True)
bio_df   = pd.concat([pd.read_csv(f) for f in bio_files], ignore_index=True)


In [6]:
def clean_region_names(df):
    for col in ["state", "district"]:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.strip()
                .str.title()
            )
    return df

# Clean all datasets
enrol_df = clean_region_names(enrol_df)
demo_df = clean_region_names(demo_df)
bio_df = clean_region_names(bio_df)

# Remove duplicates
enrol_df.drop_duplicates(inplace=True)
demo_df.drop_duplicates(inplace=True)
bio_df.drop_duplicates(inplace=True)

# Ensure year exists
for df in [enrol_df, demo_df, bio_df]:
    if "year" not in df.columns and "date" in df.columns:
        pd.to_datetime(df["date"], errors="coerce", dayfirst=True)

In [7]:
enrol_df = pd.concat([pd.read_csv(f) for f in enrolment_files], ignore_index=True)
demo_df  = pd.concat([pd.read_csv(f) for f in demo_files], ignore_index=True)
bio_df   = pd.concat([pd.read_csv(f) for f in bio_files], ignore_index=True)



In [8]:
print(enrol_df.columns)
print(demo_df.columns)


Index(['date', 'state', 'district', 'pincode', 'age_0_5', 'age_5_17', 'age_18_greater'], dtype='object')
Index(['date', 'state', 'district', 'pincode', 'demo_age_5_17', 'demo_age_17_'], dtype='object')


In [9]:
def ensure_year_column(df):
    # Case 1: already exists
    if "year" in df.columns:
        return df

    # Case 2: date column exists
    if "date" in df.columns:
        df["year"] = pd.to_datetime(
            df["date"],
            errors="coerce",
            dayfirst=True
        ).dt.year
        return df

    # Case 3: month & year columns already separated
    for col in df.columns:
        if col.lower() == "year":
            df.rename(columns={col: "year"}, inplace=True)
            return df

    # Case 4: no time info → stop early with message
    raise KeyError("No 'year' or 'date' column found in dataframe")

# Apply to both datasets
enrol_df = ensure_year_column(enrol_df)
demo_df  = ensure_year_column(demo_df)


In [10]:
print("ENROLMENT columns:", enrol_df.columns)
print("DEMOGRAPHIC columns:", demo_df.columns)


ENROLMENT columns: Index(['date', 'state', 'district', 'pincode', 'age_0_5', 'age_5_17', 'age_18_greater', 'year'], dtype='object')
DEMOGRAPHIC columns: Index(['date', 'state', 'district', 'pincode', 'demo_age_5_17', 'demo_age_17_', 'year'], dtype='object')


In [11]:
print(enrol_df.columns.tolist())


['date', 'state', 'district', 'pincode', 'age_0_5', 'age_5_17', 'age_18_greater', 'year']


In [12]:
print(demo_df.columns.tolist())


['date', 'state', 'district', 'pincode', 'demo_age_5_17', 'demo_age_17_', 'year']


In [13]:
def standardize_columns(df):
    """
    Rename dataset-specific column names to standard names
    used throughout the project.
    """
    column_map = {}

    for col in df.columns:
        col_lower = col.lower()

        # Enrolment count
        if "enrol" in col_lower or "aadhaar" in col_lower or "generated" in col_lower:
            column_map[col] = "enrolment_count"

        # Update count
        elif "update" in col_lower:
            column_map[col] = "update_count"

        # Population
        elif "population" in col_lower or "pop" in col_lower:
            column_map[col] = "estimated_population"

        # Biometric failures
        elif "failure" in col_lower:
            column_map[col] = "biometric_failures"

        # Biometric attempts
        elif "attempt" in col_lower or "auth" in col_lower:
            column_map[col] = "biometric_attempts"

    df.rename(columns=column_map, inplace=True)
    return df


In [14]:
enrol_df = standardize_columns(enrol_df)
demo_df  = standardize_columns(demo_df)
bio_df   = standardize_columns(bio_df)


In [15]:
print("ENROLMENT:", enrol_df.columns)
print("DEMOGRAPHIC:", demo_df.columns)
print("BIOMETRIC:", bio_df.columns)


ENROLMENT: Index(['date', 'state', 'district', 'pincode', 'age_0_5', 'age_5_17', 'age_18_greater', 'year'], dtype='object')
DEMOGRAPHIC: Index(['date', 'state', 'district', 'pincode', 'demo_age_5_17', 'demo_age_17_', 'year'], dtype='object')
BIOMETRIC: Index(['date', 'state', 'district', 'pincode', 'bio_age_5_17', 'bio_age_17_'], dtype='object')


In [16]:
print("ENROL DF COLUMNS:")
for col in enrol_df.columns:
    print(col)


ENROL DF COLUMNS:
date
state
district
pincode
age_0_5
age_5_17
age_18_greater
year


In [17]:
print("\nDEMO DF COLUMNS:")
for col in demo_df.columns:
    print(col)



DEMO DF COLUMNS:
date
state
district
pincode
demo_age_5_17
demo_age_17_
year


In [18]:
enrol_df = enrol_df.rename(columns={
    "date": "enrolment_count"
})


In [19]:
demo_df = demo_df.rename(columns={
    "date": "estimated_population"
})


In [20]:
print("ENROLMENT columns after rename:", enrol_df.columns)
print("DEMO columns after rename:", demo_df.columns)


ENROLMENT columns after rename: Index(['enrolment_count', 'state', 'district', 'pincode', 'age_0_5', 'age_5_17', 'age_18_greater', 'year'], dtype='object')
DEMO columns after rename: Index(['estimated_population', 'state', 'district', 'pincode', 'demo_age_5_17', 'demo_age_17_', 'year'], dtype='object')


In [21]:
enrol_agg = enrol_df.groupby(
    ["state", "district", "year"],
    as_index=False
)["enrolment_count"].sum()

pop_agg = demo_df.groupby(
    ["state", "district", "year"],
    as_index=False
)["estimated_population"].sum()


In [22]:
demo_df.rename(columns={"Gender": "gender"}, inplace=True)
enrol_df.rename(columns={"Gender": "gender"}, inplace=True)


In [23]:
gender_cols = [
    c for c in demo_df.columns
    if "male" in c.lower() or "female" in c.lower()
]

demo_gender_long = demo_df.melt(
    id_vars=["state", "district", "year"],
    value_vars=gender_cols,
    var_name="gender",
    value_name="population_value"   # TEMP name
)


In [24]:
demo_gender_long["gender"] = demo_gender_long["gender"].str.contains(
    "male", case=False
).map({True: "Male", False: "Female"})


In [25]:
demo_gender_long.rename(
    columns={"population_value": "estimated_population"},
    inplace=True
)


In [27]:
enrol_gender_cols = [
    c for c in enrol_df.columns
    if "male" in c.lower() or "female" in c.lower()
]

enrol_gender_long = enrol_df.melt(
    id_vars=["state", "district", "year"],
    value_vars=enrol_gender_cols,
    var_name="gender",
    value_name="enrolment_value"   # TEMP name
)

enrol_gender_long["gender"] = enrol_gender_long["gender"].str.contains(
    "male", case=False
).map({True: "Male", False: "Female"})

enrol_gender_long.rename(
    columns={"enrolment_value": "enrolment_count"},
    inplace=True
)


In [28]:
gender_df = demo_gender_long.groupby(
    ["state", "district", "gender", "year"],
    as_index=False
)["estimated_population"].sum()

gender_enrol = enrol_gender_long.groupby(
    ["state", "district", "gender", "year"],
    as_index=False
)["enrolment_count"].sum()

gender_merge = gender_df.merge(
    gender_enrol,
    on=["state", "district", "gender", "year"],
    how="left"
).fillna(0)

gender_merge["gender_invisibility"] = (
    (gender_merge["estimated_population"] - gender_merge["enrolment_count"])
    / gender_merge["estimated_population"]
)

female_gap = gender_merge[gender_merge["gender"] == "Female"]
display(female_gap.sort_values("gender_invisibility", ascending=False).head(10))


,state,district,gender,year,estimated_population,enrolment_count,gender_invisibility


In [29]:
# Find age-related columns dynamically
age_cols = [
    c for c in enrol_df.columns
    if any(x in c.lower() for x in ["age", "0-5", "5-18", "18-60", "60"])
]

print("Detected age columns:", age_cols)


Detected age columns: ['age_0_5', 'age_5_17', 'age_18_greater']


In [30]:
enrol_age_long = enrol_df.melt(
    id_vars=["state", "district", "year"],
    value_vars=age_cols,
    var_name="age_group",
    value_name="enrolment_value"   # TEMP name
)


In [31]:
def clean_age_group(label):
    label = label.lower()
    if "0" in label and "5" in label:
        return "Children (0–5)"
    elif "5" in label and "18" in label:
        return "Children (5–18)"
    elif "18" in label and "60" in label:
        return "Adults (18–60)"
    elif "60" in label:
        return "Elderly (60+)"
    else:
        return "Unknown"

enrol_age_long["age_group"] = enrol_age_long["age_group"].apply(clean_age_group)


In [32]:
enrol_age_long.rename(
    columns={"enrolment_value": "enrolment_count"},
    inplace=True
)


In [33]:
if "update_count" in enrol_df.columns:
    update_agg = enrol_df.groupby(
        ["state", "district", "year"],
        as_index=False
    )["update_count"].sum()

    enrol_age_long = enrol_age_long.merge(
        update_agg,
        on=["state", "district", "year"],
        how="left"
    )
else:
    enrol_age_long["update_count"] = 0


In [34]:
age_df = enrol_age_long.groupby(
    ["state", "district", "age_group", "year"],
    as_index=False
)[["enrolment_count", "update_count"]].sum()

display(age_df.head())


,state,district,age_group,year,enrolment_count,update_count
0,100000,100000,Children (0–5),2025,0,0
1,100000,100000,Unknown,2025,218,0
2,Andaman & Nicobar Islands,Andamans,Children (0–5),2025,70,0
3,Andaman & Nicobar Islands,Andamans,Unknown,2025,5,0
4,Andaman & Nicobar Islands,Nicobars,Children (0–5),2025,1,0


In [35]:
def ensure_year_column(df):
    """
    Ensures a 'year' column exists.
    Tries: year → date → fails loudly if impossible.
    """
    if "year" in df.columns:
        return df

    if "date" in df.columns:
        df["year"] = pd.to_datetime(
            df["date"],
            errors="coerce",
            dayfirst=True
        ).dt.year
        return df

    raise KeyError("No 'year' or 'date' column found in biometric dataset")
    

bio_df = ensure_year_column(bio_df)


In [36]:
print(bio_df.columns)


Index(['date', 'state', 'district', 'pincode', 'bio_age_5_17', 'bio_age_17_', 'year'], dtype='object')


In [37]:
def standardize_biometric_columns(df):
    col_map = {}

    for col in df.columns:
        col_l = col.lower()

        if "fail" in col_l:
            col_map[col] = "biometric_failures"

        elif "attempt" in col_l or "auth" in col_l:
            col_map[col] = "biometric_attempts"

    df = df.rename(columns=col_map)
    return df


bio_df = standardize_biometric_columns(bio_df)


In [38]:
print("BIO DF COLUMNS:")
for col in bio_df.columns:
    print(col)


BIO DF COLUMNS:
date
state
district
pincode
bio_age_5_17
bio_age_17_
year
